# NT409 — Week 3: First Steps with Qiskit
## Single-Qubit Systems, Gates and Circuits

**Faculty of Computer Networks and Communications — University of Information Technology, VNU-HCM**

---

Ten exercises on a single qubit, all solvable with what was covered in class this week and all running
on a local simulator. No quantum hardware and no IBM Quantum account are needed.

### How to work through it

1. Run the setup cell first; everything later depends on it.
2. Each exercise has a cell containing `# TODO`. Replace the `None` placeholders with your own code.
3. Run the cell. A short `PASS` / `FAIL` report tells you whether your answer is correct.
4. Work out what you expect on paper *before* running a cell. The prediction is the point of the exercise,
   not the output.

You can run this notebook locally (Python 3.10 or newer) or in Google Colab.


## 0. Setup

If Qiskit is not installed yet, remove the `#` in the next cell and run it once.
On Google Colab you always need to run it.

In [ ]:
# %pip install "qiskit>=1.0" matplotlib pylatexenc

Now import what we need and define a small helper, `check`, that reports whether an answer is correct.

In [ ]:
import numpy as np
import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_bloch_multivector, plot_histogram

print("Qiskit version:", qiskit.__version__)

sampler = StatevectorSampler()


def check(label, ok, hint=""):
    """Print a one-line PASS/FAIL report."""
    if ok:
        print("PASS   " + label)
    else:
        print("FAIL   " + label + (("   hint: " + hint) if hint else ""))


def run_counts(circuit, shots=1000):
    """Run a circuit that ends with measure_all() and return its counts dictionary."""
    result = sampler.run([circuit], shots=shots).result()
    return result[0].data.meas.get_counts()


print("Setup complete.")

## 1. Qubit states

A single-qubit state is a vector of two complex amplitudes,

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1 ,$$

and in Qiskit it is a `Statevector`, built either from a label or from its amplitudes.

In [ ]:
# Worked example
zero = Statevector.from_label("0")                        # |0>
plus = Statevector.from_label("+")                        # (|0> + |1>)/sqrt(2)
custom = Statevector([1 / np.sqrt(2), 1j / np.sqrt(2)])   # (|0> + i|1>)/sqrt(2)

print("|0>     amplitudes:", zero.data)
print("|+>     amplitudes:", plus.data)
print("|+>     probabilities:", plus.probabilities_dict())
print("custom  probabilities:", custom.probabilities_dict())

### Exercise 1

Build the state

$$|\psi\rangle = \tfrac{\sqrt3}{2}\,|0\rangle + \tfrac12\,|1\rangle$$

and compute its measurement probabilities with `probabilities_dict()`.
Write the two numbers down on paper before running the cell.

In [ ]:
# TODO: replace None
psi = None                 # the state above
probs = None               # its probabilities, as a dictionary

check("psi is normalised", psi is not None and abs(np.linalg.norm(psi.data) - 1) < 1e-9)
check("P(0) = 0.75", probs is not None and abs(probs["0"] - 0.75) < 1e-9, "P(0) = |alpha|^2")
check("P(1) = 0.25", probs is not None and abs(probs["1"] - 0.25) < 1e-9)

### Exercise 2

Multiplying a whole state by a phase factor changes nothing physical, while a phase on one amplitude
alone gives a genuinely different state. `Statevector.equiv(other)` returns `True` exactly when two
vectors describe the same physical state.

Build $-|-\rangle$, the state $|-\rangle$ with every amplitude multiplied by $-1$, and compare it
with $|-\rangle$ and with $|+\rangle$.

In [ ]:
minus = Statevector.from_label("-")
plus = Statevector.from_label("+")

# TODO: replace None
minus_neg = None            # the state -|->

check("minus_neg is the same state as |->",
      minus_neg is not None and minus_neg.equiv(minus), "multiply the data of |-> by -1")
check("minus_neg is NOT the same state as |+>",
      minus_neg is not None and not minus_neg.equiv(plus))
check("|-> and -|-> give the same probabilities",
      minus_neg is not None and np.allclose(minus_neg.probabilities(), minus.probabilities()))

## 2. The Bloch sphere

Every single-qubit state can be written with two angles,

$$|\psi\rangle = \cos\tfrac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\tfrac{\theta}{2}\,|1\rangle ,$$

where $\theta$ is the polar angle and $\varphi$ is the relative phase.

In [ ]:
# Worked example: four states and where they sit
for label in ["0", "1", "+", "r"]:       # 'r' is |+i> = (|0> + i|1>)/sqrt(2)
    sv = Statevector.from_label(label)
    print(label, "->", np.round(sv.data, 3))

plot_bloch_multivector(Statevector.from_label("r"))

### Exercise 3

Write a function that returns $(\theta, \varphi)$ for a single-qubit state, using

$$\theta = 2\arccos|\alpha| , \qquad \varphi = \arg\beta - \arg\alpha .$$

Wrap $\varphi$ into $[0, 2\pi)$ with the modulo operator `%`, since `np.angle` returns values in $(-\pi, \pi]$.

In [ ]:
def bloch_angles(sv):
    alpha, beta = sv.data
    # TODO: replace None
    theta = None
    phi = None
    return theta, phi


th, ph = bloch_angles(Statevector.from_label("+"))
check("|+> has theta = pi/2", th is not None and abs(th - np.pi / 2) < 1e-9)
check("|+> has phi = 0", ph is not None and abs(ph % (2 * np.pi)) < 1e-9)

th, ph = bloch_angles(Statevector.from_label("l"))          # |-i>
check("|-i> has phi = 3pi/2", ph is not None and abs(ph - 3 * np.pi / 2) < 1e-6,
      "use % (2 * np.pi) to move a negative angle into [0, 2pi)")

th, ph = bloch_angles(Statevector([np.sqrt(3) / 2, 1 / 2]))
check("the state of Exercise 1 has theta = pi/3", th is not None and abs(th - np.pi / 3) < 1e-9)

## 3. Gates

A gate is a 2x2 unitary matrix. In Qiskit you add it to a circuit (`qc.x(0)`, `qc.h(0)`, ...) and apply
that circuit to a state with `sv.evolve(qc)`. `Operator(qc).data` returns the matrix of a circuit.

In [ ]:
# Worked example: X turns |0> into |1>
qc = QuantumCircuit(1)
qc.x(0)

print("X|0> =", Statevector.from_label("0").evolve(qc).data)
print("matrix of the circuit:\n", np.round(Operator(qc).data, 3))
print(qc.draw())

### Exercise 4

Confirm on the computer what you can already work out by hand:

$$X|0\rangle = |1\rangle, \qquad H|0\rangle = |+\rangle, \qquad Z|+\rangle = |-\rangle .$$

Build one circuit per gate and apply it with `.evolve(...)`.

In [ ]:
zero = Statevector.from_label("0")
plus = Statevector.from_label("+")

qc_x, qc_h, qc_z = QuantumCircuit(1), QuantumCircuit(1), QuantumCircuit(1)
# TODO: add one gate to each circuit
# qc_x.?(0)
# qc_h.?(0)
# qc_z.?(0)

check("X|0> = |1>", zero.evolve(qc_x).equiv(Statevector.from_label("1")))
check("H|0> = |+>", zero.evolve(qc_h).equiv(plus))
check("Z|+> = |->", plus.evolve(qc_z).equiv(Statevector.from_label("-")))

### Exercise 5

Verify three identities numerically, by comparing matrices:

$$H X H = Z, \qquad S\cdot S = Z, \qquad T\cdot T = S .$$

For each one, build a circuit applying the gates in the stated order and compare `Operator(...).data`
with the matrix of the single gate on the right-hand side.

In [ ]:
qc_hxh, qc_ss, qc_tt = QuantumCircuit(1), QuantumCircuit(1), QuantumCircuit(1)
# TODO: fill the three circuits (H then X then H; S twice; T twice)

qc_z, qc_s = QuantumCircuit(1), QuantumCircuit(1)
qc_z.z(0)
qc_s.s(0)

check("HXH = Z", np.allclose(Operator(qc_hxh).data, Operator(qc_z).data),
      "an empty circuit has the identity as its matrix, so nothing was added yet")
check("S.S = Z", np.allclose(Operator(qc_ss).data, Operator(qc_z).data))
check("T.T = S", np.allclose(Operator(qc_tt).data, Operator(qc_s).data))

### Exercise 6

The rotation gate $R_y(\theta)$ turns $|0\rangle$ into

$$R_y(\theta)\,|0\rangle = \cos\tfrac{\theta}{2}\,|0\rangle + \sin\tfrac{\theta}{2}\,|1\rangle .$$

Choose the angle that prepares the state of Exercise 1 and build the circuit with `qc.ry(theta, 0)`.

In [ ]:
qc_prep = QuantumCircuit(1)
# TODO: pick the right angle and add the gate
theta = None
# qc_prep.ry(theta, 0)

target = Statevector([np.sqrt(3) / 2, 1 / 2])
check("theta = pi/3", theta is not None and abs(theta - np.pi / 3) < 1e-9, "cos(theta/2) = sqrt(3)/2")
check("the circuit prepares the target state",
      Statevector.from_label("0").evolve(qc_prep).equiv(target))

## 4. Circuits and the order of gates

A circuit is read from left to right, while the matrices multiply from right to left: the circuit
A, then B, then C means $C\,B\,A\,|\psi\rangle$. `Statevector(qc)` applies a circuit to $|0\rangle$.

In [ ]:
# Worked example
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)
qc.h(0)

final = Statevector(qc)
print(qc.draw())
print("final amplitudes:", np.round(final.data, 3))
print("same state as |-i>?", final.equiv(Statevector.from_label("l")))

### Exercise 7

Starting from $|0\rangle$, build two circuits:

- circuit A: X first, then H;
- circuit B: H first, then X.

Predict both final states before running the cell, and note that they differ even though the two
circuits contain the same gates.

In [ ]:
qc_a, qc_b = QuantumCircuit(1), QuantumCircuit(1)
# TODO: build the two circuits

state_a = Statevector(qc_a)
state_b = Statevector(qc_b)

check("circuit A gives |->", state_a.equiv(Statevector.from_label("-")))
check("circuit B gives |+>", state_b.equiv(Statevector.from_label("+")))
check("the two states are different", not state_a.equiv(state_b))

### Exercise 8

Build the circuit H, then Z, then H, and show two things: its final state is $|1\rangle$, and its matrix
is the matrix of the X gate. `Operator(a).equiv(Operator(b))` compares two circuits up to a global phase.

In [ ]:
qc_hzh = QuantumCircuit(1)
# TODO: build the circuit

qc_x_only = QuantumCircuit(1)
qc_x_only.x(0)

check("the final state is |1>", Statevector(qc_hzh).equiv(Statevector.from_label("1")))
check("the circuit is the X gate", Operator(qc_hzh).equiv(Operator(qc_x_only)))
print(qc_hzh.draw())

## 5. Measurement and shots

A measurement returns one bit per run, so probabilities appear only as frequencies over many runs, called
shots. Add `measure_all()` to a circuit and run it with the sampler.

Hardware measures in the $\{|0\rangle, |1\rangle\}$ basis only. To measure in the
$\{|+\rangle, |-\rangle\}$ basis, apply H first: outcome 0 then means $|+\rangle$ and outcome 1 means $|-\rangle$.

In [ ]:
# Worked example: the same state measured in two bases
qc_z = QuantumCircuit(1)
qc_z.h(0)                  # prepare |+>
qc_z.measure_all()

qc_x = QuantumCircuit(1)
qc_x.h(0)                  # prepare |+>
qc_x.h(0)                  # rotate the X basis onto the Z basis
qc_x.measure_all()

counts_z = run_counts(qc_z, shots=1000)
print("Z basis:", counts_z)
print("X basis:", run_counts(qc_x, shots=1000))
plot_histogram(counts_z)

### Exercise 9

For $p = 1/2$ the standard error of an estimated probability is

$$\sigma = \sqrt{\frac{p(1-p)}{N}} = \frac{1}{2\sqrt{N}} .$$

Run the $|+\rangle$ circuit for 100, 1 000 and 10 000 shots, estimate $P(0)$ each time, and compare the
error you actually got with $2\sigma$.

About one run in twenty falls outside the two-sigma band, so an occasional `FAIL` here is not a mistake.
Run the cell a few times and watch how the numbers move.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.measure_all()

for shots in [100, 1000, 10000]:
    counts = run_counts(qc, shots=shots)
    # TODO: replace None
    p0_estimate = None          # counts of "0" divided by the number of shots
    sigma = None                # 1 / (2 * sqrt(shots))

    if p0_estimate is None or sigma is None:
        print("shots =", shots, "-> fill in the TODOs")
        continue
    error = abs(p0_estimate - 0.5)
    print(f"shots = {shots:6d}   P(0) = {p0_estimate:.3f}   error = {error:.3f}   2 sigma = {2 * sigma:.3f}")
    check(f"{shots} shots: the estimate lies inside the two-sigma band", error <= 2 * sigma)

### Exercise 10

The function below prepares one of the four states $|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$,
chosen from the number you pass to it. Identify which one it is using **measurements only**.

A measurement in the Z basis separates $|0\rangle$ and $|1\rangle$ from $|+\rangle$ and $|-\rangle$;
a measurement in the X basis then separates $|+\rangle$ from $|-\rangle$. Do not print the state vector:
on real hardware you could not do that.

In [ ]:
def mystery_circuit(student_number):
    """Return a circuit preparing one of |0>, |1>, |+>, |->, without saying which."""
    qc = QuantumCircuit(1)
    k = (student_number * 7 + 3) % 4
    if k in (1, 3):
        qc.x(0)
    if k >= 2:
        qc.h(0)
    return qc


my_number = 2025                        # replace with your own student number
my_circuit = mystery_circuit(my_number)

# TODO: measure my_circuit in the Z basis and in the X basis, then decide.
# Use my_circuit.copy(), add the gates you need, call measure_all() and run_counts().

answer = None                           # write "0", "1", "+" or "-"

truth = ["0", "1", "+", "-"][(my_number * 7 + 3) % 4]
check("the mystery state is identified", answer == truth,
      "start with the Z basis: a certain result means |0> or |1>")

## What to hand in

- The notebook, run from top to bottom without errors, with every check reporting `PASS`
  (Exercise 9 may fail by chance; say so in a comment if it does).
- For Exercises 1, 7 and 8, add a short comment stating the prediction you made **before** running the code.
- Use your own student number in Exercise 10.
- Save the file with your name and student number in its file name.

Qiskit documentation: <https://quantum.cloud.ibm.com/docs>


---

### Note

Save the notebook after running every cell (so the outputs are visible) as `.ipynb`.

Author: phatpt@uit.edu.vn
